In [ ]:
# Cell 1: Environment and Imports
import os, json, re, hashlib, shutil, zipfile, sys, subprocess
from pathlib import Path
from datetime import datetime
try:
    from openai import OpenAI
except Exception:
    subprocess.run([sys.executable,'-m','pip','install','--quiet','openai>=1.0.0'],check=True); from openai import OpenAI
if not os.getenv('OPENAI_API_KEY'):
    raise SystemExit('ERROR: OPENAI_API_KEY not set')
client = OpenAI()
ROOT = Path('.').resolve(); print('Env OK', ROOT)


In [ ]:
# Cell 2: Configuration
for d in ['content/research','content/drafts','content/edits','content/outline','content/style','build','dist','logs','references','cache','content/research_inputs']:
    Path(d).mkdir(parents=True, exist_ok=True)
book_spec = {
 'title':'Working Title','subtitle':'Optional Subtitle','author':'Your Name','audience':'Primary reader profile','goal':'Outcome for the reader','genre':'Nonfiction','tone':'Clear, friendly, concise','reading_level':'General','target_length_words':20000,'chapters':8,
 'outline_constraints':['Every chapter ends with a checklist and 3 exercises'],
 'style_guide':{'voice':'Second person, active','formatting':'Markdown H2 for sections, H3 for subsections; bullet lists are OK','citations':'Numeric placeholders [n] with end-of-chapter list','terminology':['Define key terms once and reuse consistently']},
 'research_policy':{'enabled':False,'sources_allowed':['peer-reviewed','publisher docs','government'],'sources_disallowed':['unattributed personal blogs'],'citation_format':'numeric'},
 'constraints':{'originality':'No long verbatim quotes from non-public sources','copyright':'No paywalled copy-paste','localization':'en-US'},
 'export':{'docx':True,'epub':False,'pdf':False}
}
pipeline_config = {
 'RUN_COST_CAP_USD': float(os.getenv('RUN_COST_CAP_USD', 3.0)),
 'CHAPTER_COST_CAP_USD': float(os.getenv('CHAPTER_COST_CAP_USD', 0.25)),
 'MODEL_ID_FAST': os.getenv('MODEL_ID_FAST','gpt-3.5-turbo'),
 'MODEL_ID_THINK': os.getenv('MODEL_ID_THINK','gpt-4-turbo'),
 'TEMPERATURE': 0.2, 'RESEARCH_ENABLED': False, 'SAMPLE_RUN_CHAPTERS': 2, 'FULL_RUN': True, 'ULTRA_BUDGET_MODE': False
}
print('Config OK', pipeline_config['MODEL_ID_FAST'])


In [ ]:
# Cell 3: Utilities
def read_text(p): p=Path(p); return p.read_text(encoding='utf-8') if p.exists() else ''
def write_text(p,s): p=Path(p); p.parent.mkdir(parents=True,exist_ok=True); p.write_text(s,encoding='utf-8')
def read_json(p): p=Path(p); return json.loads(p.read_text(encoding='utf-8')) if p.exists() else None
def write_json(p,d): p=Path(p); p.parent.mkdir(parents=True,exist_ok=True); p.write_text(json.dumps(d,indent=2,ensure_ascii=False),encoding='utf-8')
def has_file(p): return Path(p).is_file()
def stamp(p): write_text(p, 'checkpoint: '+datetime.utcnow().isoformat()+'Z')
def sanitize_md(t): t=(t or '').replace('
','
').strip(); t=re.sub('^```(json|markdown)?','',t,flags=re.I|re.M); t=re.sub('```$','',t,flags=re.M); return t.strip()
def count_words(t): return len((t or '').split())
def approx_tokens(t): return max(1, int(len((t or ''))/4))
def sha1(s): return hashlib.sha1((s or '').encode('utf-8')).hexdigest()
class CostCapExceededException(Exception): pass
class CostTracker:
  PR={'gpt-3.5-turbo':{'in':0.0005,'out':0.0015},'gpt-4-turbo':{'in':0.01,'out':0.03},'gpt-4o-mini':{'in':0.00015,'out':0.0006},'default':{'in':0.001,'out':0.003}}
  def __init__(s,cap): s.cap=float(cap); s.spent=0.0; s.log=[]
  def price(s,m): return next((v for k,v in s.PR.items() if k!='default' and k in m), s.PR['default'])
  def est(s,m,pt,ct): p=s.price(m); return (pt/1000)*p['in']+(ct/1000)*p['out']
  def can(s,a): return s.spent+float(a) <= s.cap+1e-9
  def spend(s,l,a): a=float(a);
    if not s.can(a): raise CostCapExceededException('Cap hit before '+l+f' need {a:.4f}, left {s.cap-s.spent:.4f}'); s.spent+=a; s.log.append({'label':l,'reserve':round(a,6),'t':datetime.utcnow().isoformat()+'Z'})
  def recon(s,l,est,act): d=float(act)-float(est);
    if d>0 and not s.can(d): raise CostCapExceededException('Cap hit reconciling '+l+f' +{d:.4f}'); s.spent+=max(0,d); s.log.append({'label':l,'est':round(est,6),'act':round(act,6),'delta':round(d,6),'t':datetime.utcnow().isoformat()+'Z'})
  def summary(s): return {'total_spent_usd':round(s.spent,6),'run_cap_usd':round(s.cap,6),'remaining_usd':round(max(0,s.cap-s.spent),6),'log_items':len(s.log)}
print('Utils ready')


In [ ]:
# Cell 4: Agents Wiring
PMROUTER_PROMPT='You are PMRouter. Keep costs under caps. Enforce checkpoints. Minimize tokens. Log every step.'
AUTHOR_AGENT_PROMPT='You are AuthorAgent. Draft clear structured text per style. Add [n] placeholders for claims. Keep concise.'
EDITOR_AGENT_PROMPT='You are EditorAgent. Light fact-check notes, enforce tone/heading rules, remove filler, add [n] markers. Output clean draft + claims.'
RESEARCH_AGENT_PROMPT='You are ResearchAgent. Low-cost, prefer local sources. If none, produce questions and suggested source types only.'

def chat(model,sysm,userm,temp=0.2,max_t=800):
  try:
    r=client.chat.completions.create(model=model,messages=[{'role':'system','content':sysm},{'role':'user','content':userm}],temperature=temp,max_tokens=max_t)
    u=getattr(r,'usage',None); u={'prompt_tokens':getattr(u,'prompt_tokens',None),'completion_tokens':getattr(u,'completion_tokens',None)} if u else None
    return (r.choices[0].message.content or ''),u
  except Exception:
    import openai as O; O.api_key=os.getenv('OPENAI_API_KEY'); r=O.ChatCompletion.create(model=model,messages=[{'role':'system','content':sysm},{'role':'user','content':userm}],temperature=temp,max_tokens=max_t); u=r.get('usage'); u={'prompt_tokens':u.get('prompt_tokens'),'completion_tokens':u.get('completion_tokens')} if u else None; return r['choices'][0]['message']['content'],u

class Agent:
  def __init__(s,name,sysm,model,temp=0.2,max_t=800,cache='cache',tracker=None): s.name=name; s.sysm=sysm; s.model=model; s.temp=temp; s.max_t=max_t; s.cache=Path(cache); s.cache.mkdir(exist_ok=True,parents=True); s.tracker=tracker
  def run(s,label,prompt,max_t=None,cache_key=None):
    key=sha1(f'{s.model}
{s.sysm}
{prompt}
{cache_key or ''}'); c=s.cache/f'{label}_{key}.json'
    if c.exists(): d=json.loads(c.read_text(encoding='utf-8')); return {'text':d.get('text',''),'cached':True,'usage':d.get('usage'),'est_cost':0.0}
    pt=approx_tokens(s.sysm)+approx_tokens(prompt); ct=int((max_t or s.max_t)*0.9); est=s.tracker.est(s.model,pt,ct) if s.tracker else 0.0
    (s.tracker.spend(label+'[reserve]',est) if s.tracker else None); txt,usage=chat(s.model,s.sysm,prompt,s.temp,max_t or s.max_t)
    if usage and s.tracker: s.tracker.recon(label+'[actual]',est,s.tracker.est(s.model,usage.get('prompt_tokens') or pt,usage.get('completion_tokens') or ct))
    c.write_text(json.dumps({'text':txt,'usage':usage,'ts':datetime.utcnow().isoformat()+'Z'},ensure_ascii=False),encoding='utf-8'); return {'text':txt,'cached':False,'usage':usage,'est_cost':est}

class PMRouter:
  def __init__(s,spec,cfg,tracker): s.spec=spec; s.cfg=cfg; s.tr=tracker; m=cfg['MODEL_ID_FAST']; s.author=Agent('Author',AUTHOR_AGENT_PROMPT,m,cfg['TEMPERATURE'],1000,'cache',tracker); s.editor=Agent('Editor',EDITOR_AGENT_PROMPT,m,cfg['TEMPERATURE'],900,'cache',tracker); s.research=Agent('Research',RESEARCH_AGENT_PROMPT,m,0.2,600,'cache',tracker) if cfg.get('RESEARCH_ENABLED') else None; Path('logs').mkdir(exist_ok=True)
  def log(s,label,meta): meta=dict(meta or {}); meta['label']=label; meta['time']=datetime.utcnow().isoformat()+'Z'; p=Path('logs')/('{}_{}.json'.format(datetime.utcnow().strftime('%Y%m%dT%H%M%S'), label.replace(' ','_'))); p.parent.mkdir(exist_ok=True); p.write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding='utf-8')
print('Agents ready')


In [ ]:
# Cell 5: Style Guide and Glossary
def gen_style(spec):
  g=f"# Style Guide
- Voice: {spec['style_guide']['voice']}
- Formatting: {spec['style_guide']['formatting']}
- Citations: {spec['style_guide']['citations']}
- Terminology: "+', '.join(spec['style_guide']['terminology'])+"

Rules: Short, clear paragraphs. H2/H3 only. Add [n] where needed. Tone: "+spec['tone']
  write_text('content/style/style_guide.md', g); write_json('content/style/glossary.json', {'terms':[]}); return 'content/style/style_guide.md','content/style/glossary.json'
print('Style writers ready')


In [ ]:
# Cell 6: Outline Generation
def brief(spec): return 'Title:'+spec['title']+'
Audience:'+spec['audience']+'
Goal:'+spec['goal']+'
Tone:'+spec['tone']+'
Chapters:'+str(spec['chapters'])+'
TargetWords:'+str(spec['target_length_words'])+'
Style:'+spec['style_guide']['formatting']

def parse_json_loose(t):
  t=sanitize_md(t); i=t.find('{'); j=t.rfind('}')
  if i!=-1 and j!=-1 and j>i:
    try: return json.loads(t[i:j+1])
    except Exception: return None
  try: return json.loads(t)
  except Exception: return None

def gen_outline(spec,tr,router):
  pj='content/outline/outline.json'; pm='content/outline/outline.md'
  if has_file(pj) and has_file(pm): return pj,pm
  instr='JSON outline with key "chapters" (array). Each chapter: number,title,sections(array of section titles and bullets),learning_objectives(array). Output JSON only.'
  R=router.author.run('outline', 'BRIEF
'+brief(spec)+'

'+instr, max_t=900, cache_key=str(spec.get('chapters'))); d=parse_json_loose(R['text'])
  if not d or 'chapters' not in d: raise RuntimeError('Invalid outline JSON'); write_json(pj,d)
  lines=['# Outline: '+spec['title'],'']
  for ch in d['chapters']: lines.append('## Chapter '+str(ch.get('number'))+': '+str(ch.get('title','')))
  write_text(pm, '
'.join(lines)); return pj,pm
print('Outline gen ready')


In [ ]:
# Cell 7: Per-Chapter Loop
def ch_dir(ch,sub): d=Path(f'content/{sub}/{ch:02d}'); d.mkdir(parents=True,exist_ok=True); return d

def outline_ch(pj,ch): d=read_json(pj) or {}
  
  for it in d.get('chapters',[]):
    if int(it.get('number',-1))==int(ch): return it
  raise KeyError('Chapter not in outline')

def parse_editor_blocks(t):
  def ex(tag):
    import re
    m=re.search('<'+tag+'>\s*([\s\S]*?)\s*</'+tag+'>',t)
    return (m.group(1) if m else '').strip()
  md,claims,notes=ex('DRAFT_EDITED_MD'),ex('CLAIMS_REPORT_JSON'),ex('CONTINUITY_NOTES_MD')
  try:
    c=json.loads(sanitize_md(claims)) if claims else {'notes':'none'}
  except Exception:
    c={'raw':claims}
  return sanitize_md(md),c,sanitize_md(notes)

def process_chapter(ch,cfg,tracker,router,pj):
  ch=int(ch); cap=float(cfg['CHAPTER_COST_CAP_USD']); start=tracker.spent; dr=ch_dir(ch,'research'); dd=ch_dir(ch,'drafts'); de=ch_dir(ch,'edits')
  b=dr/'brief.md'; s=dr/'sources.json'
  if cfg.get('RESEARCH_ENABLED') and (not b.exists() or not s.exists()):
    oc=outline_ch(pj,ch); note='locals present' if (Path('content/research_inputs')/f'{ch:02d}').exists() else 'no locals'
    R=router.research.run(f'ch{ch}_research','OUTLINE:'+json.dumps(oc)+'
NOTE:'+note+'
Blocks: <BRIEF_MD>..</BRIEF_MD> and <SOURCES_JSON>{...}</SOURCES_JSON>. No web.', max_t=500, cache_key=str(oc)[:400]) if router.research else {'text':'<BRIEF_MD>Research off</BRIEF_MD><SOURCES_JSON>{"enabled":false}</SOURCES_JSON>'}
    t=R['text']
    import re
    bm=re.search('<BRIEF_MD>\s*([\s\S]*?)\s*</BRIEF_MD>',t); sm=re.search('<SOURCES_JSON>\s*([\s\S]*?)\s*</SOURCES_JSON>',t)
    write_text(b, sanitize_md(bm.group(1)) if bm else '')
    try: write_json(s, json.loads(sanitize_md(sm.group(1))) if sm else {'enabled':False})
    except Exception: write_json(s, {'raw': sanitize_md(sm.group(1)) if sm else ''})
  dp=dd/'draft.md'
  if not dp.exists():
    oc=outline_ch(pj,ch); tw=max(450,min(800,int(book_spec['target_length_words']//max(1,book_spec['chapters'])))); br=read_text(b) if b.exists() else ''
    prompt='OUTLINE:'+json.dumps(oc)+'
BRIEF:'+br[:800]+f"
Write ONLY markdown for '## Chapter {ch}: {oc.get('title','')}' with H3 sections, Checklist, 3 Exercises. Add [n] where needed. ~{tw} words."
    R=router.author.run(f'ch{ch}_draft', prompt, max_t=900, cache_key=str(oc)[:400]); write_text(dp, sanitize_md(R['text']))
  if (tracker.spent-start)>=cap or tracker.summary()['remaining_usd']<0.05:
    ep,decp,nop=de/'draft_edited.md',de/'claims_report.json',de/'continuity_notes.md'; write_text(ep,read_text(dp)); write_json(decp,{'warning':'editor skipped budget','chapter':ch}); write_text(nop,'Skipped'); return
  ep,decp,nop=de/'draft_edited.md',de/'claims_report.json',de/'continuity_notes.md'
  if not (ep.exists() and decp.exists() and nop.exists()):
    d=read_text(dp); d=d if len(d)<12000 else d[:12000]
    instr='INPUT:'+d+'
Output blocks: <DRAFT_EDITED_MD>..</DRAFT_EDITED_MD><CLAIMS_REPORT_JSON>{"claims":["..."]}</CLAIMS_REPORT_JSON><CONTINUITY_NOTES_MD>..</CONTINUITY_NOTES_MD>. Keep style; no URLs.'
    R=router.editor.run(f'ch{ch}_edit', instr, max_t=800, cache_key=sha1(d)); md,cl,nt=parse_editor_blocks(R['text']); write_text(ep, md or read_text(dp)); write_json(decp,cl); write_text(nop,nt)
print('Chapter processor ready')


In [ ]:
# Cell 8: Assembly
def assemble_book(spec):
  fm=['# '+spec['title'], ('## '+spec['subtitle']) if spec.get('subtitle') else '', '**Author:** '+spec['author'],'','---','']
  chs=[]; toc=[]; n=int(spec['chapters'])
  for ch in range(1,n+1):
    e=Path(f'content/edits/{ch:02d}/draft_edited.md'); d=Path(f'content/drafts/{ch:02d}/draft.md'); t=read_text(e) if e.exists() else (read_text(d) if d.exists() else f'## Chapter {ch}: (missing)
TODO'); chs.append(t.strip()); tl=next((ln for ln in t.splitlines() if ln.startswith('## ')), f'Chapter {ch}'); toc.append({'chapter':ch,'title_line':tl})
  md='

'.join([x for x in fm if x]+chs); write_text('build/book.md',md); write_json('build/toc.json',toc); return 'build/book.md','build/toc.json'
print('Assembly ready')


In [ ]:
# Cell 9: QA
def run_qa(spec):
  rpt={'checks':{},'warnings':[]}
  p=Path('build/book.md'); ok=p.exists() and p.stat().st_size>0; rpt['checks']['book_exists']=bool(ok)
  if not ok: write_json('dist/qa_report.json',rpt); return rpt
  t=read_text(p); wc=count_words(t); tgt=int(spec.get('target_length_words',20000)); low,hi=int(tgt*0.9),int(tgt*1.1); within=(low<=wc<=hi); rpt['checks']['word_count_ok']=bool(within)
  if not within: rpt['warnings'].append('Word count '+str(wc)+' vs target '+str(tgt))
  present=all(('## Chapter '+str(ch)+':') in t for ch in range(1,int(spec['chapters'])+1)); rpt['checks']['all_chapters_present']=bool(present)
  if not present: rpt['warnings'].append('Some chapters missing/labels')
  notodo=('TODO' not in t and 'FIXME' not in t); rpt['checks']['no_todo_fixme']=bool(notodo)
  if not notodo: rpt['warnings'].append('TODO/FIXME remain')
  head_ok=('####' not in t); rpt['checks']['heading_levels_ok']=bool(head_ok)
  if not head_ok: rpt['warnings'].append('Headings exceed H3')
  cites_ok=True
  if spec.get('genre','').lower()=='nonfiction' and pipeline_config.get('RESEARCH_ENABLED'):
    cites_ok=('[n]' in t)
    if not cites_ok: rpt['warnings'].append('No [n] markers while research enabled')
  rpt['checks']['citations_present_if_research']=bool(cites_ok)
  rpt['outcome']='PASS' if not rpt['warnings'] else 'PASS_WITH_WARNINGS'
  write_json('dist/qa_report.json',rpt); return rpt
print('QA ready')


In [ ]:
# Cell 10: Export
def minimal_docx(p,note='DOCX export unavailable; install python-docx. See dist/book.md'):
  p=Path(p); p.parent.mkdir(parents=True,exist_ok=True)
  with zipfile.ZipFile(p,'w',compression=zipfile.ZIP_DEFLATED) as z:
    z.writestr('[Content_Types].xml','<?xml version="1.0" encoding="UTF-8" standalone="yes"?><Types xmlns="http://schemas.openxmlformats.org/package/2006/content-types"><Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/><Default Extension="xml" ContentType="application/xml"/><Override PartName="/word/document.xml" ContentType="application/vnd.openxmlformats-officedocument.wordprocessingml.document.main+xml"/></Types>')
    z.writestr('_rels/.rels','<?xml version="1.0" encoding="UTF-8" standalone="yes"?><Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships"><Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/officeDocument" Target="word/document.xml"/></Relationships>')
    z.writestr('word/document.xml','<?xml version="1.0" encoding="UTF-8" standalone="yes"?><w:document xmlns:w="http://schemas.openxmlformats.org/wordprocessingml/2006/main"><w:body><w:p><w:r><w:t>'+note.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')+'</w:t></w:r></w:p></w:body></w:document>')

def export_deliverables():
  src=Path('build/book.md'); dst=Path('dist/book.md'); dst.parent.mkdir(parents=True,exist_ok=True); shutil.copyfile(src,dst) if src.exists() else write_text(dst,'# Book (missing)')
  try:
    import docx; d=docx.Document(); md=read_text(src) if src.exists() else '# Book (missing)'
    for ln in md.splitlines():
      d.add_heading(ln[4:],3) if ln.startswith('### ') else (d.add_heading(ln[3:],2) if ln.startswith('## ') else (d.add_heading(ln[2:],1) if ln.startswith('# ') else d.add_paragraph(ln)))
    d.save('dist/book.docx')
  except Exception:
    minimal_docx('dist/book.docx')
  return str(dst),'dist/book.docx'
print('Export ready')


In [ ]:
# Cell 11: Manifest
def manifest(tr,outcome):
  def scan(d):
    root=Path(d); items=[]
    if not root.exists(): return items
    for p in sorted(root.rglob('*')):
      if p.is_file(): items.append({'path':str(p).replace('\','/'),'size_bytes':p.stat().st_size,'mtime':datetime.utcfromtimestamp(p.stat().st_mtime).isoformat()+'Z'})
    return items
  m={'outcome':outcome,'generated_at':datetime.utcnow().isoformat()+'Z','files':{'build':scan('build'),'dist':scan('dist')},'cost_summary':tr.summary()}
  write_json('dist/manifest.json',m); return m
print('Manifest ready')


In [ ]:
# Cell 12: Run-All / Rerun
def make_router(): tr=CostTracker(pipeline_config['RUN_COST_CAP_USD']); rt=PMRouter(book_spec,pipeline_config,tr); return rt,tr

def run_all():
  rt,tr=make_router(); out='PASS'
  try:
    gen_style(book_spec); pj,pm=gen_outline(book_spec,tr,rt); total=int(book_spec['chapters']); chs=list(range(1,total+1)) if pipeline_config.get('FULL_RUN',True) else list(range(1,min(total,int(pipeline_config.get('SAMPLE_RUN_CHAPTERS',2)))+1))
    for ch in chs:
      try: process_chapter(ch,pipeline_config,tr,rt,pj)
      except CostCapExceededException as e: write_text('logs/last_error.txt',str(e)); out='ABORTED_COST_CAP'; break
    assemble_book(book_spec); qa=run_qa(book_spec); out='PASS_WITH_WARNINGS' if qa.get('warnings') else out; export_deliverables()
  except CostCapExceededException as e: write_text('logs/last_error.txt',str(e)); out='ABORTED_COST_CAP'
  except Exception as e: write_text('logs/last_error.txt','FAILED_STEP: '+str(e)); out='FAILED_STEP'
  finally: manifest(tr,out)
  print('Run done',out); return out

def rerun_chapter(n): n=int(n)
  for sub in ['research','drafts','edits']:
    d=Path(f'content/{sub}/{n:02d}')
    if d.exists():
      for p in d.glob('*'):
        if p.is_file(): p.unlink()
  rt,tr=make_router(); pj='content/outline/outline.json'
  if not Path(pj).exists(): gen_outline(book_spec,tr,rt)
  process_chapter(n,pipeline_config,tr,rt,pj); assemble_book(book_spec); run_qa(book_spec); export_deliverables(); manifest(tr,'PASS'); print('Rerun done',n)

def resume(): return run_all()
print('Controls ready')


In [ ]:
# Cell 13: Demonstration Mode
pipeline_config['SAMPLE_RUN_CHAPTERS']=2; pipeline_config['FULL_RUN']=False; pipeline_config['RESEARCH_ENABLED']=False
outcome=run_all(); m=read_json('dist/manifest.json') or {}; print('Outcome:',m.get('outcome')); print('Cost:',m.get('cost_summary'))

def tree(d): d=Path(d)
for p in sorted(d.rglob('*')):
  if p.is_file(): print(str(d)+'/'+str(p.relative_to(d)).replace('\','/'), '('+str(p.stat().st_size)+' bytes)')
print('
Build:'); tree(Path('build')); print('
Dist:'); tree(Path('dist'))
